# M4.S5 — Asynchronous Practice 4
## Part 1 — Guided Jupyter Lab
### Data parallelism: shard → synchronize → measure → explain

**Guided route: about 55–60 minutes**

This lab consolidates the distributed-ML material from Module 4.

```text
M4.S2  HPC & machine learning
          ↓
       data parallelism · batches · shards · gradients

M4.S3  distributed ML workflows
          ↓
       communication · orchestration · serving · recovery

M4.S5  guided data-parallel experiment
          ↓
       PROVE correctness → RUN on Slurm → MEASURE → EXPLAIN scaling
```

Today we use **real PyTorch DistributedDataParallel (DDP)** with a small synthetic classification problem.

The experiment is deliberately small. The goal is not to build the best model. The goal is to understand the **systems behaviour**:

> **DATA → SHARD → LOCAL COMPUTE → ALL-REDUCE → SAME UPDATE → MEASURE → EXPLAIN**

You will:

1. verify that sharding covers the dataset without overlap;
2. connect global batch size to local batch size;
3. prove that a two-worker DDP update matches a serial global-batch update;
4. run real 1-worker and 2-worker jobs through **Slurm**;
5. compare runtime, throughput, synchronization and model correctness;
6. record the configuration and results for reproducibility.

After this notebook, **Part 2 in Blackboard** asks you to apply the same reasoning independently to a different distributed-ML experiment.

> **Notebook build: M4S5-2026-v1**


## 0 — Before we start: what makes this a fair experiment?

We will change **one main factor**:

> **number of DDP workers**

We keep the following fixed:

- same synthetic dataset;
- same model architecture;
- same number of epochs;
- same **global batch size**;
- same optimizer and learning rate;
- same Slurm allocation size;
- same software environment;
- same random seeds.

The important distinction is:

```text
GLOBAL BATCH = all samples processed before one synchronized optimizer update

LOCAL BATCH = the part of the global batch processed by one worker

local batch = global batch / number of workers
```

### PREDICT

If the global batch is 512:

- 1 worker → local batch = ?
- 2 workers → local batch = ?
- 4 workers → local batch = ?

And: **does adding workers guarantee that training becomes faster?**


In [ ]:
from pathlib import Path
import json, os, re, shutil, socket, subprocess, sys, time, platform
from statistics import median

OUT = Path("m4s5_outputs")
OUT.mkdir(exist_ok=True)

GLOBAL_BATCH = 512
SAMPLES = 16384
EPOCHS = 4
FEATURES = 96
CLASSES = 6

print("Host:", platform.node())
print("Python:", sys.version.split()[0])
print("Visible CPUs:", os.cpu_count())
print("sbatch:", shutil.which("sbatch"))
print("squeue:", shutil.which("squeue"))
print("sinfo:", shutil.which("sinfo"))
print("torchrun in current environment:", shutil.which("torchrun"))
print()
print("Experiment:")
print(" samples =", SAMPLES)
print(" epochs =", EPOCHS)
print(" global batch =", GLOBAL_BATCH)
print(" features =", FEATURES)
print(" classes =", CLASSES)


## 1 — Mini-example: what exactly is a shard?

A shard is a **non-overlapping subset of the dataset** assigned to one worker.

For data parallel training we want:

- every training sample assigned to a worker;
- no accidental duplication;
- no missing samples.

Run this small example before the real training job.


In [ ]:
sample_ids = list(range(20))
workers = 4

shards = {
    rank: sample_ids[rank::workers]
    for rank in range(workers)
}

for rank, ids in shards.items():
    print(f"worker {rank}: {ids}")

flattened = [x for rank in range(workers) for x in shards[rank]]
print("\nall samples covered:", sorted(flattened) == sample_ids)
print("duplicates:", len(flattened) - len(set(flattened)))


### OBSERVE

Sharding changes **where the data is processed**.

It does **not** by itself combine gradients or keep model replicas synchronized.

That requires communication.

---

## 2 — Mini-example: global batch versus local batch

A common source of confusion is changing the number of workers **and** accidentally changing the effective batch size.

For today's experiment the **global batch stays fixed at 512**.


In [ ]:
for world_size in (1, 2, 4, 8):
    assert GLOBAL_BATCH % world_size == 0
    local_batch = GLOBAL_BATCH // world_size
    print(
        f"{world_size:>2} worker(s): "
        f"local batch {local_batch:>3} × {world_size} = global batch {GLOBAL_BATCH}"
    )


### Why keep the global batch fixed?

Because we want the comparison to answer a clean question:

> **What changes when more workers cooperate on the same training job?**

If we changed worker count, global batch, learning rate and model at the same time, interpretation would be much harder.

---

## 3 — Write the real PyTorch DDP program

The next cell creates a real training program.

It has two modes:

### `equivalence`
Two workers each process half of one global batch.

DDP averages their gradients with collective communication.

We compare the result against a serial model trained on the whole global batch.

### `train`
The same training workload is run with different worker counts.

The program records:

- world size;
- global and local batch;
- training wall time;
- examples/s;
- accuracy;
- replica checksum spread;
- gradient payload size;
- measured All-Reduce time for a payload equal to the model gradients.


In [ ]:
ddp_program = r'''
import argparse, json, os, socket, time
import torch
import torch.distributed as dist
from torch import nn
from torch.nn.parallel import DistributedDataParallel as DDP

def setup():
    if not dist.is_available():
        raise RuntimeError("torch.distributed is unavailable")
    dist.init_process_group(backend="gloo")
    torch.set_num_threads(1)
    return dist.get_rank(), dist.get_world_size()

def cleanup():
    if dist.is_initialized():
        dist.destroy_process_group()

def model_checksum(model):
    return sum(float(p.detach().double().sum()) for p in model.parameters())

def gathered_scalar(value):
    t = torch.tensor([value], dtype=torch.float64)
    out = [torch.zeros_like(t) for _ in range(dist.get_world_size())]
    dist.all_gather(out, t)
    return [float(v.item()) for v in out]

def make_model(features, classes):
    return nn.Sequential(
        nn.Linear(features, 64),
        nn.ReLU(),
        nn.Linear(64, classes),
    )

def equivalence(rank, world):
    features, classes, global_batch = 32, 4, 256
    if global_batch % world:
        raise ValueError("global batch must be divisible by world size")

    gen = torch.Generator().manual_seed(2026)
    X = torch.randn(global_batch, features, generator=gen)
    true_w = torch.randn(features, classes, generator=gen)
    y = (X @ true_w).argmax(dim=1)

    torch.manual_seed(77)
    template = make_model(features, classes)
    initial = {k: v.detach().clone() for k, v in template.state_dict().items()}

    local_batch = global_batch // world
    lo, hi = rank * local_batch, (rank + 1) * local_batch

    model = make_model(features, classes)
    model.load_state_dict(initial)
    ddp = DDP(model)
    opt = torch.optim.SGD(ddp.parameters(), lr=0.1)
    opt.zero_grad(set_to_none=True)
    loss = nn.functional.cross_entropy(ddp(X[lo:hi]), y[lo:hi])
    loss.backward()
    opt.step()

    dist.barrier()
    checks = gathered_scalar(model_checksum(model))
    replica_spread = max(checks) - min(checks)

    if rank == 0:
        ref = make_model(features, classes)
        ref.load_state_dict(initial)
        ref_opt = torch.optim.SGD(ref.parameters(), lr=0.1)
        ref_opt.zero_grad(set_to_none=True)
        ref_loss = nn.functional.cross_entropy(ref(X), y)
        ref_loss.backward()
        ref_opt.step()

        max_diff = max(
            float((a.detach() - b.detach()).abs().max())
            for a, b in zip(model.parameters(), ref.parameters())
        )
        result = {
            "kind": "equivalence",
            "world_size": world,
            "global_batch": global_batch,
            "local_batch": local_batch,
            "max_parameter_difference": max_diff,
            "replica_checksum_spread": replica_spread,
            "pass": bool(max_diff < 1e-6 and replica_spread < 1e-9),
        }
        print("M4S5_RESULT " + json.dumps(result))

def train(rank, world, epochs, global_batch, samples, features, classes):
    if global_batch % world:
        raise ValueError("global batch must be divisible by world size")
    if samples % global_batch:
        raise ValueError("samples must be divisible by global batch")
    if samples % world:
        raise ValueError("samples must be divisible by world size")

    gen = torch.Generator().manual_seed(2027)
    X = torch.randn(samples, features, generator=gen)
    true_w = torch.randn(features, classes, generator=gen)
    noise = 0.15 * torch.randn(samples, classes, generator=gen)
    y = (X @ true_w + noise).argmax(dim=1)

    torch.manual_seed(99)
    model = make_model(features, classes)
    ddp = DDP(model)
    opt = torch.optim.SGD(ddp.parameters(), lr=0.12)

    local_batch = global_batch // world
    steps_per_epoch = samples // global_batch
    shard_size = samples // world
    shard_start = rank * shard_size
    local_X = X[shard_start:shard_start + shard_size]
    local_y = y[shard_start:shard_start + shard_size]

    with torch.no_grad():
        _ = ddp(local_X[:local_batch])
    dist.barrier()

    t0 = time.perf_counter()
    last_loss = 0.0
    for _ in range(epochs):
        for step in range(steps_per_epoch):
            lo = step * local_batch
            hi = lo + local_batch
            opt.zero_grad(set_to_none=True)
            out = ddp(local_X[lo:hi])
            loss = nn.functional.cross_entropy(out, local_y[lo:hi])
            loss.backward()
            opt.step()
            last_loss = float(loss.detach())
    dist.barrier()

    local_elapsed = time.perf_counter() - t0
    elapsed_t = torch.tensor([local_elapsed], dtype=torch.float64)
    dist.all_reduce(elapsed_t, op=dist.ReduceOp.MAX)
    elapsed = float(elapsed_t.item())

    checks = gathered_scalar(model_checksum(model))
    replica_spread = max(checks) - min(checks)

    num_params = sum(p.numel() for p in model.parameters())
    gradient_bytes = num_params * 4
    buf = torch.ones(num_params, dtype=torch.float32)
    for _ in range(3):
        dist.all_reduce(buf)
    dist.barrier()
    t1 = time.perf_counter()
    rounds = 20
    for _ in range(rounds):
        dist.all_reduce(buf)
    dist.barrier()
    comm_local = (time.perf_counter() - t1) / rounds
    comm_t = torch.tensor([comm_local], dtype=torch.float64)
    dist.all_reduce(comm_t, op=dist.ReduceOp.MAX)
    comm_s = float(comm_t.item())

    if rank == 0:
        with torch.no_grad():
            accuracy = float((model(X).argmax(dim=1) == y).float().mean())
        result = {
            "kind": "train",
            "host": socket.gethostname(),
            "world_size": world,
            "samples": samples,
            "features": features,
            "classes": classes,
            "epochs": epochs,
            "global_batch": global_batch,
            "local_batch": local_batch,
            "steps_per_epoch": steps_per_epoch,
            "training_seconds": elapsed,
            "examples_per_second": samples * epochs / elapsed,
            "final_local_loss": last_loss,
            "accuracy": accuracy,
            "replica_checksum_spread": replica_spread,
            "gradient_bytes": gradient_bytes,
            "allreduce_gradient_ms": 1000.0 * comm_s,
            "backend": dist.get_backend(),
        }
        print("M4S5_RESULT " + json.dumps(result))

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices=["equivalence", "train"], default="train")
    ap.add_argument("--epochs", type=int, default=4)
    ap.add_argument("--global-batch", type=int, default=512)
    ap.add_argument("--samples", type=int, default=16384)
    ap.add_argument("--features", type=int, default=96)
    ap.add_argument("--classes", type=int, default=6)
    args = ap.parse_args()
    rank, world = setup()
    try:
        if args.mode == "equivalence":
            equivalence(rank, world)
        else:
            train(rank, world, args.epochs, args.global_batch, args.samples, args.features, args.classes)
    finally:
        cleanup()

if __name__ == "__main__":
    main()
'''

Path("m4s5_ddp_lab.py").write_text(ddp_program)
print("Wrote m4s5_ddp_lab.py")
print("Lines:", len(ddp_program.splitlines()))


## 4 — Slurm helpers

As in the previous asynchronous practice, the **real experiment runs through Slurm**.

The notebook will:

1. submit a job;
2. wait for it to finish;
3. read the output;
4. extract the JSON result.

Optional cluster tools or GPU access are not required for this guided route.


In [ ]:
def clean_submit_env():
    env = os.environ.copy()
    for key in ("SLURM_MEM_PER_CPU", "SLURM_MEM_PER_GPU", "SLURM_MEM_PER_NODE"):
        env.pop(key, None)
    return env

def submit(script, exports=None):
    if not shutil.which("sbatch"):
        raise RuntimeError(
            "sbatch is unavailable. Run this notebook on the SciTech JupyterHub."
        )
    cmd = ["sbatch", "--parsable"]
    if exports:
        joined = ",".join(f"{k}={v}" for k, v in exports.items())
        cmd += [f"--export=ALL,{joined}"]
    cmd += [script]
    p = subprocess.run(
        cmd, capture_output=True, text=True, env=clean_submit_env()
    )
    if p.returncode != 0:
        raise RuntimeError(p.stderr or p.stdout)
    job = p.stdout.strip().split(";")[0]
    print("Submitted:", job, exports or {})
    return job

def wait_job(job, timeout=600):
    start = time.time()
    while time.time() - start < timeout:
        p = subprocess.run(
            ["squeue", "-h", "-j", str(job), "-o", "%T"],
            capture_output=True, text=True
        )
        state = p.stdout.strip()
        if not state:
            time.sleep(2)
            return True
        print("Job", job, state)
        time.sleep(4)
    raise TimeoutError(f"Timed out waiting for job {job}")

def read_output(job):
    path = Path(f"m4s5-{job}.out")
    for _ in range(15):
        if path.exists():
            text = path.read_text(errors="replace")
            print(text)
            return text
        time.sleep(1)
    raise FileNotFoundError(path)

def parse_result(text):
    lines = [ln for ln in text.splitlines() if ln.startswith("M4S5_RESULT ")]
    if not lines:
        raise ValueError("No M4S5_RESULT line found")
    return json.loads(lines[-1].split(" ", 1)[1])

print("Slurm helpers ready")


## 5 — Create one controlled Slurm job

The same allocation is used for the required experiments.

We request four CPU cores for the job, but PyTorch uses:

- one process for the 1-worker run;
- two processes for the 2-worker run.

Each process is restricted to one CPU thread so that worker count remains visible in the experiment.

The PyTorch module below is the same environment used in the previous distributed-ML session.


In [ ]:
slurm_script = f"""#!/bin/bash
#SBATCH --job-name=m4s5-ddp
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=4G
#SBATCH --time=00:08:00
#SBATCH --output=m4s5-%j.out

set -euo pipefail
cd "$SLURM_SUBMIT_DIR"

unset SLURM_MEM_PER_CPU
unset SLURM_MEM_PER_GPU
unset SLURM_MEM_PER_NODE

module load PyTorch/2.1.2-foss-2023a

export OMP_NUM_THREADS=1
export MKL_NUM_THREADS=1
export OPENBLAS_NUM_THREADS=1

WORLD_SIZE=${WORLD_SIZE:-1}
MODE=${MODE:-train}

echo "=== CONFIGURATION ==="
echo "JOB_ID=$SLURM_JOB_ID"
echo "HOST=$(hostname)"
echo "DATE=$(date -Is)"
echo "WORLD_SIZE=$WORLD_SIZE"
echo "MODE=$MODE"
echo "GLOBAL_BATCH={GLOBAL_BATCH}"
echo "SAMPLES={SAMPLES}"
echo "EPOCHS={EPOCHS}"
echo "--- modules ---"
module -t list 2>&1 || true
echo "=== END CONFIGURATION ==="
echo

torchrun --standalone --nproc_per_node=$WORLD_SIZE \
    m4s5_ddp_lab.py \
    --mode $MODE \
    --epochs {EPOCHS} \
    --global-batch {GLOBAL_BATCH} \
    --samples {SAMPLES} \
    --features {FEATURES} \
    --classes {CLASSES}
"""

Path("m4s5_run.sbatch").write_text(slurm_script)
print(Path("m4s5_run.sbatch").read_text())


# 6 — PROVE correctness before discussing performance

This is the most important correctness experiment in the notebook.

Two workers:

```text
global batch = 256

worker 0 → 128 samples → local gradient
worker 1 → 128 samples → local gradient
                       ↓
                   All-Reduce
                       ↓
               same averaged gradient
```

We compare that DDP update with a serial model that processes all 256 samples at once.

### PREDICT

Should the updated parameters be exactly or almost exactly the same?

**Yes**, apart from tiny floating-point differences.

Run the next cell.


In [ ]:
equiv_job = submit("m4s5_run.sbatch", {
    "WORLD_SIZE": 2,
    "MODE": "equivalence",
})
wait_job(equiv_job)
equiv_text = read_output(equiv_job)
equiv = parse_result(equiv_text)

print("\nParsed result:")
print(json.dumps(equiv, indent=2))

assert equiv["pass"], "DDP update did not match the serial reference"


### What proves correctness?

Look for:

- `world_size = 2`;
- `local_batch = 128`;
- very small `max_parameter_difference`;
- `replica_checksum_spread = 0` or extremely close to zero;
- `pass = true`.

This shows two different things:

1. **the distributed update agrees with the serial global-batch update**;
2. **both model replicas remain synchronized**.

Only now do we move to performance.

---

# 7 — Real experiment: 1 worker versus 2 workers

We now run the same workload twice.

```text
Run A: world size = 1
Run B: world size = 2
```

Everything else stays fixed.

### PREDICT BEFORE RUNNING

Which outcome do you expect?

A. 2 workers are almost exactly 2× faster  
B. 2 workers are faster, but by less than 2×  
C. 2 workers are about the same  
D. 2 workers are slower

Any answer is possible in principle.

The result depends on the balance between:

> **useful computation ↔ synchronization ↔ process overhead ↔ data movement**


In [ ]:
job1 = submit("m4s5_run.sbatch", {"WORLD_SIZE": 1, "MODE": "train"})
job2 = submit("m4s5_run.sbatch", {"WORLD_SIZE": 2, "MODE": "train"})

wait_job(job1)
wait_job(job2)

text1 = read_output(job1)
text2 = read_output(job2)

r1 = parse_result(text1)
r2 = parse_result(text2)

print("\n1-worker result:")
print(json.dumps(r1, indent=2))
print("\n2-worker result:")
print(json.dumps(r2, indent=2))


## 8 — Compare the evidence

For strong scaling of this fixed workload:

`S₂ = T₁ / T₂`

and

`E₂ = S₂ / 2`

But speedup alone is not enough.

We also inspect:

- examples/s;
- local batch size;
- accuracy;
- replica synchronization;
- gradient payload;
- All-Reduce latency.


In [ ]:
t1 = r1["training_seconds"]
t2 = r2["training_seconds"]
speedup = t1 / t2
efficiency = speedup / 2.0

rows = [
    ("workers", r1["world_size"], r2["world_size"]),
    ("local batch", r1["local_batch"], r2["local_batch"]),
    ("training seconds", f"{t1:.4f}", f"{t2:.4f}"),
    ("examples/s", f"{r1['examples_per_second']:.0f}", f"{r2['examples_per_second']:.0f}"),
    ("accuracy", f"{r1['accuracy']:.3f}", f"{r2['accuracy']:.3f}"),
    ("replica checksum spread", f"{r1['replica_checksum_spread']:.2e}", f"{r2['replica_checksum_spread']:.2e}"),
    ("gradient bytes", r1["gradient_bytes"], r2["gradient_bytes"]),
    ("All-Reduce gradient ms", f"{r1['allreduce_gradient_ms']:.4f}", f"{r2['allreduce_gradient_ms']:.4f}"),
]

print(f"{'metric':30s} {'1 worker':>16s} {'2 workers':>16s}")
print("-"*66)
for metric, a, b in rows:
    print(f"{metric:30s} {str(a):>16s} {str(b):>16s}")

print()
print(f"speedup S2    = {speedup:.3f}x")
print(f"efficiency E2 = {efficiency:.3f}")


### OBSERVE, then explain

Answer these in your own words:

1. Did two workers reduce training time?
2. Was the speedup close to 2×?
3. Did the **global batch** change?
4. Did the **local batch** change?
5. Are the two DDP replicas synchronized?
6. How large is one model-gradient payload?
7. How long does one All-Reduce of that payload take?
8. What overheads could explain imperfect scaling?

A perfectly valid result is:

> **Two workers are slower for this small problem.**

That is not a failed experiment.

It is evidence that the workload is too small, communication is too expensive, or process overhead dominates.

In HPC, **more workers are not automatically better**.


## 9 — A small scaling model

A useful mental model is:

```text
time per training step
≈ local compute
+ synchronization
+ other overhead
```

When we add workers:

```text
local compute per worker ↓

but

synchronization does not disappear
```

If the computation becomes too small relative to communication, scaling efficiency falls.

### Question

What kind of change would make 2-worker DDP more likely to help?

Possible answers include:

- a larger model;
- more computation per sample;
- a larger problem;
- faster interconnect;
- less frequent synchronization;
- better overlap of compute and communication.


In [ ]:
# A simple thought experiment — NOT a measurement.
compute_1 = 8.0
comm_1 = 0.0
compute_2 = compute_1 / 2
comm_2 = 1.5

step_1 = compute_1 + comm_1
step_2 = compute_2 + comm_2

print("Illustrative model only")
print(f"1 worker step: {step_1:.1f} ms")
print(f"2 worker step: {step_2:.1f} ms")
print(f"predicted speedup: {step_1/step_2:.2f}x")
print()
print("Do not confuse this model with the measured Slurm results above.")


## 10 — Reproducibility record

A performance result without context is weak evidence.

We save:

- job IDs;
- software-facing configuration;
- dataset size;
- worker counts;
- batch sizes;
- timings;
- throughput;
- correctness evidence;
- communication measurement.

This is the minimum information needed to explain what was actually run.


In [ ]:
record = {
    "session": "M4.S5",
    "practice": "Practice 4 - guided data-parallel experiment",
    "created_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "jupyter_host": platform.node(),
    "configuration": {
        "samples": SAMPLES,
        "epochs": EPOCHS,
        "global_batch": GLOBAL_BATCH,
        "features": FEATURES,
        "classes": CLASSES,
        "slurm_partition": "cpu",
        "slurm_cpus_per_task": 4,
        "torch_module": "PyTorch/2.1.2-foss-2023a",
    },
    "equivalence_job_id": equiv_job,
    "equivalence": equiv,
    "training_jobs": {
        "1_worker": {"job_id": job1, "result": r1},
        "2_workers": {"job_id": job2, "result": r2},
    },
    "derived": {
        "speedup_2_workers": speedup,
        "efficiency_2_workers": efficiency,
    },
}

record_path = OUT / "m4s5_guided_record.json"
record_path.write_text(json.dumps(record, indent=2))

print(record_path.read_text())
print("\nSaved:", record_path)


## 11 — Optional extension: try 4 workers

This is **optional**.

Only run it if the cluster is available and you have time.

The global batch remains 512, so:

```text
4 workers → local batch = 128
```

Before running, predict whether 4 workers will improve or worsen the result.

If you run it, compare:

- `T1`, `T2`, `T4`;
- `S2`, `S4`;
- `E2`, `E4`;
- All-Reduce time;
- examples/s.

A scaling curve is useful even when performance gets worse.


In [ ]:
RUN_OPTIONAL_4_WORKERS = False

if RUN_OPTIONAL_4_WORKERS:
    job4 = submit("m4s5_run.sbatch", {"WORLD_SIZE": 4, "MODE": "train"})
    wait_job(job4)
    text4 = read_output(job4)
    r4 = parse_result(text4)

    s4 = r1["training_seconds"] / r4["training_seconds"]
    e4 = s4 / 4.0

    print(json.dumps(r4, indent=2))
    print(f"\nS4 = {s4:.3f}x")
    print(f"E4 = {e4:.3f}")
else:
    print("Optional 4-worker experiment skipped.")
    print("Set RUN_OPTIONAL_4_WORKERS = True if you want to run it.")


# 12 — Guided-lab exit ticket

Without looking back, complete the chain:

### 1. Data distribution
One dataset is divided across DDP workers using **__________**.

### 2. Local work
Each worker owns the same **__________** but processes different data.

### 3. Communication
During backpropagation, local gradients are combined using **__________**.

### 4. Correctness
After synchronization, model replicas should remain **__________**.

### 5. Performance
Adding workers helps only if saved computation is larger than **__________**.

<details>
<summary><strong>Check your answer</strong></summary>

1. **sharding**
2. **model replica**
3. **collective communication / All-Reduce**
4. **synchronized**
5. **communication + synchronization + process/data-movement overhead**

</details>

---

## Four things to remember

**Data parallelism replicates the model and splits the data.**

**The global batch is distributed across local worker batches.**

**Correct distributed execution must be proved before performance is interpreted.**

**More workers help only when useful parallel work outweighs coordination overhead.**


# Part 2 — Blackboard assignment

You have now completed the **guided** experiment.

The Blackboard assignment uses the **same evidence discipline** but a **different distributed-ML problem**.

You will work independently through SSH + Slurm and produce evidence for:

> **WORKFLOW → CONFIGURATION → RESULT → PERFORMANCE OBSERVATION → SCALABILITY/LIMITATION → REPRODUCIBILITY**

Do **not** copy the numerical results from this notebook.

Follow the separate Practice 4 instructions in Blackboard.
